In [1]:
import os
from dotenv import load_dotenv

# LangChain & LangGraph Imports
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("🚨 OPENAI_API_KEY not found! Check your .env file.")

# Enable LangSmith Tracing (Highly recommended for Agentic RAG)
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Nexteir_Second_Brain_Lab"

print("✅ Environment loaded. Tracing enabled.")

✅ Environment loaded. Tracing enabled.


In [3]:
# 1. Extract: Load the PDF
pdf_path = "../data/NEXTIER_Internship_Bugs.pdf"
loader = PyPDFLoader(pdf_path)
docs = loader.load()
print(f"📄 Loaded {len(docs)} pages from the internship logs.")

# 2. Transform: Split text into optimal chunks for embedding
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=300, # 30% overlap ensures context isn't lost between chunks
    add_start_index=True 
)
splits = text_splitter.split_documents(docs)
print(f"🧩 Split into {len(splits)} chunks.")

📄 Loaded 217 pages from the internship logs.
🧩 Split into 498 chunks.


In [4]:
# 3. Load: Create embeddings and store in ChromaDB
persist_directory = "./chroma_db"
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Initialize Chroma and ingest the chunks
vectorstore = Chroma.from_documents(
    documents=splits, 
    embedding=embeddings,
    persist_directory=persist_directory
)

# Test the raw retrieval mechanism
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
test_query = "React Native Yarn vs NPM duplicate key error"
test_results = retriever.invoke(test_query)

print(f"🔍 Vector test successful. Retrieved {len(test_results)} snippets for the test query.")
# print(test_results[0].page_content) # Uncomment to see the raw chunk

/Users/victorloucii/agentic_ai_engineer/Unified-Knowledge-Agent/.venv/lib/python3.12/site-packages/langsmith/client.py:538: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(
Failed to multipart ingest runs: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=019d7829-e0a1-7331-b19c-b2e0eb254aa9,id=019d7829-e0a1-7331-b19c-b2e0eb254aa9


🔍 Vector test successful. Retrieved 3 snippets for the test query.


Failed to send compressed multipart ingest: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=019d7829-e0a1-7331-b19c-b2e0eb254aa9,id=019d7829-e0a1-7331-b19c-b2e0eb254aa9


In [5]:
@tool
def search_internship_history(query: str) -> str:
    """
    Search the Nexteir Internship logs. This is a comprehensive 'Second Brain' 
    containing debugging fixes (//problemXX), architectural decisions, 
    and useful CLI/Terminal commands (//useful command).
    """
    # 1. Multi-Query Expansion: Let the LLM "think" of better search terms
    # This is what makes it "Lazy Search" friendly.
    expansion_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    search_variants = expansion_llm.invoke(
        f"Generate 3 diverse search queries to find information about '{query}' "
        "in technical internship logs. Focus on keywords, problem IDs, and file paths. "
        "Respond with only the queries, one per line."
    ).content.split("\n")

    # Add the original query to the list
    all_queries = [query] + [q.strip() for q in search_variants if q.strip()]
    
    # 2. Execute a broad search across all variants
    all_docs = []
    for q in all_queries:
        all_docs.extend(retriever.invoke(q))

    # 3. Deduplicate (so we don't show the same chunk twice)
    unique_docs = {doc.page_content: doc for doc in all_docs}.values()

    # 4. Format with the Strict Rules
    formatted_context = "\n\n---\n\n".join(
        [f"Page {doc.metadata.get('page', 'Unknown')}:\n{doc.page_content}" for doc in unique_docs]
    )
    
    return formatted_context

print("🛠️ Tool 'search_internship_history' defined successfully.")
print(f"Tool Description given to LLM: {search_internship_history.description}")

🛠️ Tool 'search_internship_history' defined successfully.
Tool Description given to LLM: Search the Nexteir Internship logs. This is a comprehensive 'Second Brain' 
containing debugging fixes (//problemXX), architectural decisions, 
and useful CLI/Terminal commands (//useful command).


In [6]:
# Initialize the model
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Bind the tool to the LLM
tools = [search_internship_history]
llm_with_tools = llm.bind_tools(tools)

# Ask a specific question that requires the tool
test_message = HumanMessage(content="I'm getting an Android layout collapse in my React Native app. How did we fix this during the internship?")
response = llm_with_tools.invoke([test_message])

# Check if the AI decided to use the tool
if response.tool_calls:
    print("🧠 The Agent decided to use a tool!")
    for tool_call in response.tool_calls:
        print(f"-> Tool Selected: {tool_call['name']}")
        print(f"-> Search Query Generated by AI: {tool_call['args']['query']}")
else:
    print("⚠️ The Agent answered directly without using the tool.")
    print(response.content)

Failed to send compressed multipart ingest: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=019d782a-1084-7c90-be95-2c93bfc2dc40,id=019d782a-1084-7c90-be95-2c93bfc2dc40


🧠 The Agent decided to use a tool!
-> Tool Selected: search_internship_history
-> Search Query Generated by AI: Android layout collapse React Native


Failed to send compressed multipart ingest: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=019d782a-1084-7c90-be95-2c93bfc2dc40,id=019d782a-1084-7c90-be95-2c93bfc2dc40
